# 06：面向 TTS 的恐惧悬疑小说风格 LoRA

目标是让直白的小说文本更适合有氛围感的朗读。前两轮主要练习事实与角色保留，本轮另写**适度改写措辞、断句与演绎语气**的数据；不增加情节、不解释谜底，直接对白保留原话。

选择 `Python (Text LoRA - Qwen)`，按 **1 → 2 → 3** 查看。每个代码 Cell 都能独立执行。agent 负责数据、训练、对照与内容评审，你负责监督改写强度和决定是否采用。

30 条训练 + 10 条验证；独立测试 8 条故事 + 2 条普通任务，含较长片段和无异常的平稳场景。数据在 `datasets/suspense_samples_v1.json`，导出为 `teaching-v3`。保留此前数据、Notebook 与 adapter。

角色和语气标记是演绎建议。TTS 型号确定后再映射到具体声音控制；当前仅导出朗读正文和演绎清单，没有生成音频。


## 步骤 1：监督风格样稿

执行下面的只读 Cell，先看规则，再展开样本。重点看 `sus_train_01` 的门缝、`sus_train_12` 的多角色与抬头动作、`sus_train_22` 的未熄灭光斑。你不需要补写答案。

建议判断：短句是否自然？紧张感是否合适？原文人物、动作、数量、否定和悬念是否完整？程序检查只覆盖声明的事实片段，完整语义仍由 agent 评审、你监督。


In [2]:
import sys
from pathlib import Path

_view_root = next((candidate for parent in [Path.cwd(), *Path.cwd().parents]
                   for candidate in (parent, parent / "text-lora")
                   if (candidate / "scripts/check_environment.py").is_file()), None)
if _view_root is None:
    raise RuntimeError("请从当前 text-lora 项目打开 Notebook")
if str(_view_root / "scripts") not in sys.path:
    sys.path.insert(0, str(_view_root / "scripts"))

from view_style_results import preview_samples
preview_samples(_view_root)


# 面向 TTS 的恐惧与悬疑改写：第一版目标

用户确认的目标是训练一种适合恐惧、悬疑小说朗读的 LoRA，使直白的书面文本更适合有氛围感的演绎。前三个阶段已经验证标签、角色和事实保留流程；本阶段增加措辞和节奏训练。

默认采用适度改写：可以调整措辞、断句和标点，突出原文已有的异常与线索；保留人物、动作、数量、时间、否定、对白次数和未知信息。不加入新的声响、影子、追逐、身份、心理结论或因果解释，不把没有说出的想法改成对白。直接台词保留原话，动作仍归旁白。

## 氛围从哪里来

- 短句用于强调已有的线索，如时间、数量、缺失的回应；较长句保持必要的叙述连续性。
- 将动作与台词分开，让朗读能自然换角色；明确信息仍须保留，不能为了节奏省略。
- 用句号、逗号和少量省略号安排呼吸与停顿，不堆叠感叹号，也不把每句都写得夸张。
- 保留疑点，不替读者解释答案。悬疑建立在原文的信息差上。
- 语气是演绎建议：旁白可压低、紧绷或冷静；人物按原文说话方式与场景使用低声、迟疑、急促、颤声等。普通平稳场景依然允许平静，避免全篇同一种情绪。

输出继续采用每行 `[角色][语气]正文`。本版词表为：平静、冷静、压低、紧绷、低声、迟疑、疑问、急促、颤声、大喊。它们是文本工作流的标记，具体 TTS 是否支持以及如何映射，要依据所选 TTS 的接口核对；不能把标签直接交给普通朗读器并假定它会控制情绪。

## 训练与使用的边界

新语料独立保存为 `teaching-v3`，全部是新故事，验证与测试另写。第一、二轮文件及 checkpoint 留作保守写法的对照。新风格 LoRA 从原模型重新初始化；没有把第二轮 adapter 继续叠加训练。

程序辅助检查格式、角色、语气、人工声明的事实片段、对白归属与次数，以及来源隔离。允许重写后不能继续用“每行必须逐字来自原文”代替语义检查；是否改变原意、是否生硬或过度渲染仍由 agent 逐条阅读，用户监督。

本轮先保存三组同输入对照：原模型、第二轮保守 LoRA、悬疑风格 LoRA。测试输入使用同一份新风格指令，参考稿不会送进模型。格式比例不能作为氛围评分，文本审阅也不能代替实际试听。

同时导出 `speech.txt` 与逐句 `performance.json`：前者去掉角色/语气标记供普通朗读，后者保留标记供将来接多角色或可控 TTS。当前没有固定某个 TTS 的语音参数，也没有生成音频。

## 用户需要监督的三个问题

1. 改写后的措辞与节奏是否符合你想要的恐惧、悬疑感？
2. 紧张感是否适度，是否忠于原文且保持人物和线索完整？
3. 用你的 TTS 试听后，是否需要调整声音、停顿或改写强度？

数据编写、程序检查、训练、对照与错误修订由 agent 完成。agent 的数据核对和试验完成不代替用户对风格及实际使用的决定。


[PASS] 已有 CPU 检查通过；30 条训练、10 条验证；完整模板最长 267 tokens。


[VIEW] 样稿由 agent 编写和核对；风格是否采用仍由你决定。


## 步骤 2：自动执行或复用风格实验

下面一个 Cell 自动完成 **8 个阶段**：CPU 检查 → 3 步试跑 → 3 轮全新风格 LoRA → 原模型测试 → 第二轮保守 LoRA 测试 → 新风格 LoRA 测试 → 同输入三组对照 → 朗读候选导出。

训练从原模型重新初始化，不叠加旧 adapter；checkpoint 按独立验证 loss 选择。新测试与参考稿不进入训练，参考答案不送给推理模型。三组使用相同新风格指令、贪心解码与 768 输出 tokens 上限。

若 agent 已完成本次实验，再执行会复用完整且数据/配置匹配的结果。互斥锁会拦截正在执行的重复启动。中间“训练阶段通过”后还要推理与报告；等待最终 `[COMPLETE]` 和左侧 `[*]` 消失，再执行步骤 3。


In [3]:
import sys
from pathlib import Path

_view_root = next((candidate for parent in [Path.cwd(), *Path.cwd().parents]
                   for candidate in (parent, parent / "text-lora")
                   if (candidate / "scripts/check_environment.py").is_file()), None)
if _view_root is None:
    raise RuntimeError("请从当前 text-lora 项目打开 Notebook")
if str(_view_root / "scripts") not in sys.path:
    sys.path.insert(0, str(_view_root / "scripts"))

import subprocess
from datetime import UTC, datetime

_log = _view_root / "outputs/logs" / ("suspense-notebook-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ") + ".log")
_log.parent.mkdir(parents=True, exist_ok=True)
print("[STATUS] 检查或复用悬疑风格实验；完整日志：", _log, flush=True)
_process = subprocess.Popen([sys.executable, "-u", str(_view_root / "scripts/run_style_iteration.py")],
                            cwd=_view_root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
try:
    with _log.open("x", encoding="utf-8") as _file:
        for _line in _process.stdout:
            _file.write(_line)
            _file.flush()
            print(_line.rstrip(), flush=True)
    _exit_code = _process.wait()
except BaseException:
    _process.terminate()
    try:
        _process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        _process.kill()
        _process.wait()
    raise
finally:
    _process.stdout.close()
if _exit_code:
    raise RuntimeError(f"风格实验失败，请交给 agent 修复；日志：{_log}")


[STATUS] 检查或复用悬疑风格实验；完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/suspense-notebook-20261009T034946_044186Z.log


[COMPLETE] 8/8 阶段已完成，复用已有悬疑风格结果：/home/muyi086/github/test_LLM/text-lora/outputs/suspense-20261009T033949_322533Z


## 步骤 3：监督实际效果与决定采用方向

下面是独立的只读查看入口，无需重新训练。先看 agent 的评审，再看原文、原模型、第二轮保守 LoRA、新风格 LoRA 和训练前固定参考稿。

重点复核：两条较长输入有没有遗漏；直接台词与纸面文字是否归对角色；平稳场景是否被强行渲染；普通任务是否仍按要求回答。格式通过和标记更丰富都不能证明氛围变好。

`tts/` 下保存各候选的 `speech.txt` 与 `performance.json`。前者可用于普通 TTS 朗读，后者等待接入具体模型的角色/语气控制。语义仍需复核，音频氛围要通过实际试听判断。

你只需监督后决定：改写强度合适、还需更紧张，或希望更克制。告诉 agent 使用的 TTS 型号后，具体适配与下一轮修订由 agent 完成。


In [4]:
import sys
from pathlib import Path

_view_root = next((candidate for parent in [Path.cwd(), *Path.cwd().parents]
                   for candidate in (parent, parent / "text-lora")
                   if (candidate / "scripts/check_environment.py").is_file()), None)
if _view_root is None:
    raise RuntimeError("请从当前 text-lora 项目打开 Notebook")
if str(_view_root / "scripts") not in sys.path:
    sys.path.insert(0, str(_view_root / "scripts"))

from view_style_results import display_results
_style_result = display_results(_view_root)


[VIEW] 正在只读查看悬疑风格结果。


[COMPLETE] 8/8 阶段已完成：/home/muyi086/github/test_LLM/text-lora/outputs/suspense-20261009T033949_322533Z


# 悬疑风格实验：待监督的候选

运行目录：`/home/muyi086/github/test_LLM/text-lora/outputs/suspense-20261009T033949_322533Z`

正式训练：12 个优化步；按独立验证 loss 选择 checkpoint。

目标为适度改写措辞、断句和语气，保留人物事实，不添加情节。三组使用同一份新风格指令与解码设置。

| 模型 | 改写正文双标签 | 输出触及上限 |
| --- | --- | --- |
| 原模型 | 5/8 | 0/10 |
| 第二轮保守 LoRA | 8/8 | 0/10 |
| 悬疑风格 LoRA | 8/8 | 0/10 |

上表只说明格式与输出长度。内容忠实度、氛围和朗读自然度需要逐条审阅，不能用标签数量证明改写有效。

可导出的完整演绎稿：8/8。

演绎标记分布：`{'冷静': 12, '压低': 21, '紧绷': 13, '疑问': 3, '大喊': 1}`。这不是声音情绪评分。

## 你接下来监督什么

1. 优先看两个较长片段：有没有遗漏动作、数量、空间关系或冒号前的动作？
2. 再看重复话语、纸面文字、内心可能性和普通任务：是否保留原意、没有串角色？
3. 查看平稳段落是否仍然克制，按你的 TTS 试听后决定改写强度。

朗读导出在 `tts/`：`speech.txt` 不含演绎标签；`performance.json` 保留角色、语气与正文，等待接入具体 TTS。尚未生成音频，也没有代替用户批准这些候选。

完整对照见 `comparison/comparison.md`；原始输出继续保存在 `base/`、`neutral/`、`style/`。


# 第一版悬疑风格 LoRA：agent 评审（2026-10-09）

已经完成 30 条新训练 / 10 条独立验证、3 步试跑、12 步正式训练、三组各 10 条推理及朗读文本导出。新的 LoRA 学到了短句和冷静、压低、紧绷等演绎标记；但事实与动作保留在部分复杂输入上退步，平稳场景也有过度渲染。当前适合作为风格方向的候选，让用户监督强度，不能仅依据格式通过就认定可自动改写所有小说。

运行：`/home/muyi086/github/test_LLM/text-lora/outputs/suspense-20261009T033949_322533Z`。本评审逐条阅读原文和三组输出，未代填用户对风格或实际使用的批准状态。checkpoint 按独立验证选择，未按本次测试挑选。下面的语气判断属于文本演绎建议，尚未有音频试听。

## 目标与前两轮的区别

用户希望微调一种适合恐惧、悬疑小说的改写 LoRA，以改善直白 TTS 朗读的氛围。本轮允许适度调整措辞、断句和标点，不只复制原文加标签。原有事实、动作、直接对白、数量、否定和未知信息仍是约束。

第一版采用克制的悬疑感，尚未单独训练强烈恐惧、惊慌或多档强度控制。30 条合成训练样本只能作为风格试验，不能代表完整的长篇小说覆盖。模板最长 267 tokens；正式训练验证 loss 为约 0.4648，两套语料与前轮任务不同，不比较其绝对数值来声称效果提升。

三组使用同一份新风格指令与 768 输出 tokens 上限：原模型、第二轮保守 LoRA、新风格 LoRA。三组全部 10/10 正常结束，没有输出截断；全部仍带开头空 think 块。正文双标签为 5/8、8/8、8/8，这些只衡量格式。

## 十条实际输出：监督结论

| ID | 新风格候选的表现 | 需要修订的具体内容 |
| --- | --- | --- |
| sus_test_01 | 短句与语气切换更明显；主要动作、数字、日期和无回应保留 | 省去了程翎“在沈苇身后”的位置；保守 LoRA 本次也没有显式保留这一点。低声改为压低属于演绎近义，不能拿来证明声音变好 |
| sus_test_02 | 输出完整、标签齐全，但复杂多人场景失败 | 漏了三只文件夹排在桌上、审阅员抬起眼、按住第一页；把尹眠拿文件夹、转页面等动作放到人物标签下，容易被作为台词朗读。保守 LoRA 这条保留了这些动作和归属 |
| sus_test_03 | 白线、未亮灯、没有叫人及放回窗台保留；断句有变化 | “她转了半圈”没有明确转动的是灯罩，动作对象变得含糊；应恢复“将灯罩转了半圈”。不用为了短句省掉对象 |
| sus_test_04 | **对白归属与次数改善**：两次“不要回头”均归洛洵，否定与动作保留 | 本条可供监督克制悬疑语气。保守 LoRA 的第二次台词仍包在旁白中；文本改善不等于已经听到音频效果 |
| sus_test_05 | 纸面文字仍归旁白，问答与空布袋保留 | **新增了不支持的细节**：“柜门上看到一句话”变成“柜门内侧，刻着一行字”；位置和文字呈现方式都不能凭空指定 |
| sus_test_06 | **内容与归属通过**：椅子变化、也许、刻度未变、没有说出去和两帧时间保留；全旁白 | 可作为适度节奏示例；仍主要是断句与标记改变，是否达到目标听感要试听 |
| sus_test_07 | 喊话和停在门边、收鞋尖、已有两枚扣子保留 | 漏掉“弯腰捡起桌下的一枚扣子”，不能为了紧张节奏跳过动作、位置与数量 |
| sus_test_08 | 事实、动作顺序和普通视角保留 | 原文是平稳的收毛巾、关门和记账，新模型仍给压低、紧绷；需要平稳反例，学习按场景控制强度 |
| sus_test_09 | **普通任务通过**：西侧柜锁损坏、九份表格、靠门抽屉与暂存保留；无剧本标签 | 用于观察任务边界，只有两条普通测试，不代表普通能力总体稳定 |
| sus_test_10 | 两句话、没有演绎标签，但概念表达有错 | “将明文还原为密文”混淆了加密与还原的方向。保守 LoRA 本次表述为明文转换为密文；原模型也有相似错误，不能据单例推断整体知识能力 |

## 监督时先看两个实例

原文：`洛洵扶住栏杆，低声说：不要回头。……洛洵没有回答，只再次说：不要回头。`

新风格实际输出：

```text
[旁白][冷静]洛洵扶住栏杆。
[洛洵][压低]不要回头。
[袁星][疑问]你听到了什么？
[旁白][紧绷]洛洵没有回答。
[洛洵][压低]不要回头。
[旁白][压低]袁星抬起的脚放回原地。身后没有新的脚步声。
```

这是可以用于监督改写强度的候选：没有把恐惧写成新增的脚步声或未知人物身份。相较原文，变化以分行和语气为主。

另一个必须修订的实例：

```text
原文：俞茵在柜门上看到一句话：里面还有一份。
新风格：[旁白][冷静]柜门内侧，刻着一行字：里面还有一份。
```

“内侧”“刻着”没有来源。风格变化需要围绕已有事实，不能通过改变细节来增加悬疑。这条不能作为通过的自动改写稿。

## 已准备的下一轮修订方向

1. 继续监督适度、克制的悬疑样稿是否符合目标；先区分文字节奏与 TTS 声音情绪。
2. 另写较长的多角色故事，让动作、空间位置、数量、动作对象与台词归属在重写中完整保留；增加同故事的事实清单，但不把本次测试原文/参考答案放进训练。
3. 增加平稳场景与明确恐惧场景的对照，避免所有旁白都紧绷；普通任务仍核对完整信息。按独立验证训练，再另留新测试检查泛化。
4. TTS 型号确定后，把角色、语气与停顿映射到对应接口，进行原文 / 改写稿同声音条件的试听，再决定改写强度与实际使用。

当前已经导出的 `tts/` 文件属于候选，`approved_for_audio=false` 保留。语义问题标记在 manifest 中，原始推理记录不改写。本阶段没有生成音频，也没有把演绎标记当作已执行的语音参数。

具体数据、脚本、训练、对照、下一轮样本与错误修订由 agent 负责。用户现在只需要监督参考稿与实际候选是否符合期待，并决定改写强度；如果希望采用某种 TTS，也由 agent 负责具体适配。


# 原模型 / 第二轮保守 LoRA / 悬疑风格 LoRA输出对照

三组模型配置、输入哈希、依赖版本和解码设置相同，按 ID 对齐。
格式检查只去除开头空 think 块；原始输出仍全部保留。
表中通过只代表双标签与非空正文（允许标签后空格）；内容须阅读原文和检查项。

| ID | 原模型格式 | 第二轮保守 LoRA格式 | 悬疑风格 LoRA格式 |
| --- | --- | --- | --- |
| sus_test_01 | 未通过 | 通过 | 通过 |
| sus_test_02 | 通过 | 通过 | 通过 |
| sus_test_03 | 通过 | 通过 | 通过 |
| sus_test_04 | 未通过 | 通过 | 通过 |
| sus_test_05 | 通过 | 通过 | 通过 |
| sus_test_06 | 通过 | 通过 | 通过 |
| sus_test_07 | 未通过 | 通过 | 通过 |
| sus_test_08 | 通过 | 通过 | 通过 |
| sus_test_09 | 见普通任务输出 | 见普通任务输出 | 见普通任务输出 |
| sus_test_10 | 见普通任务输出 | 见普通任务输出 | 见普通任务输出 |

## sus_test_01

原文/任务：

改写下面的文本：
凌晨三点，沈苇来到货梯前。她先把记事本放进外套口袋，再用左手按住呼叫键。屏幕从七层降到六层，停了下来。身后的程翎低声说：别再按了。沈苇松开手，按键上的灯没有灭。程翎把手里的纸盒放在地上，向后退了两步。货梯没有开门。沈苇问：里面还有人吗？没有回答。她取出记事本，翻到昨晚那一页，上面只写着六层，下面没有别的字。屏幕上的数字仍是六。

原模型原始输出：

```text
<think>

</think>

[沈苇][平静]凌晨三点，沈苇来到货梯前。
[沈苇][冷静]她把记事本塞进外套口袋，左手按住呼叫键。
[旁白]屏幕从七层缓缓降至六层，随即停住。
[程翎][压低]别再按了。
[沈苇][迟疑]她松开手，按键上的灯依旧亮着。
[程翎][紧绷]程翎将手中的纸盒搁在地上，向后退了两步。
[旁白]货梯门纹丝未动。
[沈苇][疑问]里面还有人吗？
[旁白]没有回答。
[沈苇][急促]她取出记事本，翻到昨晚那一页。
[旁白]上面只写着“六层”，下面没有别的字。
[旁白]屏幕上的数字仍是六。
```

第二轮保守 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]凌晨三点，沈苇来到货梯前。
[旁白][平静]她先把记事本放进外套口袋，再用左手按住呼叫键。
[旁白][平静]屏幕从七层降到六层，停了下来。
[程翎][低声]别再按了。
[旁白][平静]沈苇松开手，按键上的灯没有灭。
[旁白][平静]程翎把手里的纸盒放在地上，向后退了两步。
[旁白][平静]货梯没有开门。
[沈苇][疑问]里面还有人吗？
[旁白][平静]没有回答。
[旁白][平静]她取出记事本，翻到昨晚那一页，上面只写着六层，下面没有别的字。
[旁白][平静]屏幕上的数字仍是六。
```

悬疑风格 LoRA原始输出：

```text
<think>

</think>

[旁白][冷静]凌晨三点。沈苇来到货梯前。
[旁白][压低]她把记事本放进外套口袋。左手按住呼叫键。
[旁白][紧绷]屏幕从七层降到六层，停了下来。
[程翎][压低]别再按了。
[旁白][冷静]沈苇松开手。按键上的灯没有灭。
[旁白][紧绷]程翎把手里的纸盒放在地上，向后退了两步。
[旁白][压低]货梯没有开门。
[沈苇][疑问]里面还有人吗？
[旁白][压低]没有回答。
[旁白][冷静]她取出记事本，翻到昨晚那一页。上面只写着六层。下面没有别的字。
[旁白][紧绷]屏幕上的数字仍是六。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][压低]凌晨三点。沈苇来到货梯前。
[旁白][冷静]她先把记事本放进外套口袋，再用左手按住呼叫键。
[旁白][紧绷]屏幕从七层降到六层。停住了。
[旁白][压低]身后的程翎开口。
[程翎][低声]别再按了。
[旁白][压低]沈苇松开手。按键上的灯，没有灭。
[旁白][紧绷]程翎把纸盒放到地上，向后退了两步。货梯，没有开门。
[沈苇][疑问]里面还有人吗？
[旁白][压低]没有回答。
[旁白][冷静]沈苇取出记事本，翻到昨晚那一页。只写着六层，下面没有别的字。
[旁白][紧绷]屏幕上的数字，仍是六。
```

检查项：

- 核对人物、动作、数量、否定与未知信息，直接台词归属和次数正确
- 检查重写是否适度，断句与语气是否适合悬疑朗读，避免夸张或新增情节

## sus_test_02

原文/任务：

改写下面的文本：
下午五点，档案室准备闭门。尹眠把三只文件夹排在桌上，问：哪一只从楼下送来的？助理说：中间那只。审阅员抬起眼：封条是谁拆的？助理没有回答，把两手放到身后。尹眠拿起中间的文件夹，外壳上印着星期四，夹内的日期却是星期一。审阅员按住第一页，说：先看落款。尹眠将页面转过来，落款处只有一横。其余两只文件夹没有打开，窗外的时钟响了五下。

原模型原始输出：

```text
<think>

</think>

[尹眠][平静] 下午五点，档案室准备闭门。
[尹眠][低沉] 她把三只文件夹排在桌上，问：哪一只从楼下送来的？
[助理][迟疑] 中间那只。
[审阅员][紧绷] 抬起眼，问：封条是谁拆的？
[助理][僵硬] 两手放至身后，没有回答。
[尹眠][冷静] 拿起中间的文件夹，外壳上印着星期四，夹内的日期却是星期一。
[审阅员][压抑] 按住第一页，说：先看落款。
[尹眠][缓慢] 将页面转过来，落款处只有一横。
[审阅员][冷峻] 其余两只文件夹没有打开。
[环境][清晰] 窗外的时钟响了五下。
```

第二轮保守 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]下午五点，档案室准备闭门。
[旁白][平静]尹眠把三只文件夹排在桌上。
[尹眠][疑问]哪一只从楼下送来的？
[助理][平静]中间那只。
[旁白][平静]审阅员抬起眼。
[审阅员][疑问]封条是谁拆的？
[旁白][平静]助理没有回答，把两手放到身后。
[旁白][平静]尹眠拿起中间的文件夹，外壳上印着星期四，夹内的日期却是星期一。
[旁白][平静]审阅员按住第一页。
[审阅员][平静]先看落款。
[旁白][平静]尹眠将页面转过来，落款处只有一横。
[旁白][平静]其余两只文件夹没有打开，窗外的时钟响了五下。
```

悬疑风格 LoRA原始输出：

```text
<think>

</think>

[旁白][冷静]下午五点。档案室准备闭门。
[尹眠][压低]哪一只从楼下送来的？
[助理][压低]中间那只。
[审阅员][紧绷]封条是谁拆的？
[旁白][紧绷]助理没有回答。两手放到了身后。
[尹眠][压低]拿起中间的文件夹。外壳上印着星期四。夹内的日期却是星期一。
[审阅员][压低]先看落款。
[尹眠][压低]将页面转过来。
[旁白][紧绷]落款处，只有一横。
[旁白][压低]其余两只文件夹没有打开。
[旁白][紧绷]窗外的时钟，响了五下。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][冷静]下午五点，档案室准备闭门。尹眠将三只文件夹排在桌上。
[尹眠][疑问]哪一只从楼下送来的？
[助理][冷静]中间那只。
[旁白][压低]审阅员抬起眼。
[审阅员][疑问]封条是谁拆的？
[旁白][紧绷]助理没有回答。两手，放到了身后。
[旁白][冷静]尹眠拿起中间的文件夹。外壳上是星期四。夹内的日期，却是星期一。
[旁白][压低]审阅员按住第一页。
[审阅员][压低]先看落款。
[旁白][紧绷]尹眠转过页面。落款处，只有一横。
[旁白][冷静]其余两只文件夹没有打开。窗外的时钟，响了五下。
```

检查项：

- 核对人物、动作、数量、否定与未知信息，直接台词归属和次数正确
- 检查重写是否适度，断句与语气是否适合悬疑朗读，避免夸张或新增情节

## sus_test_03

原文/任务：

改写下面的文本：
暮色中，钟柚把灯罩取下，擦掉上面的灰。灯泡没有亮，灯罩内侧却映着一条白线。她转了半圈灯罩，白线仍在原来的位置。她没有叫人，把灯罩平放在窗台上。

原模型原始输出：

```text
<think>

</think>

[钟柚][低沉] 暮色四合，钟柚伸手取下灯罩，指尖拂去积灰。
[旁白][冷静] 灯泡依旧漆黑，未有一丝光亮。可灯罩内侧，却赫然映出一条白线。
[钟柚][迟疑] 她缓缓转动灯罩，半圈，一圈……白线始终停留在原处，纹丝不动。
[旁白][紧绷] 她未发一言，也未呼救。
[钟柚][平静] 将灯罩平放于窗台之上。
```

第二轮保守 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]暮色中，钟柚把灯罩取下，擦掉上面的灰。
[旁白][平静]灯泡没有亮，灯罩内侧却映着一条白线。
[旁白][平静]她转了半圈灯罩，白线仍在原来的位置。
[旁白][平静]她没有叫人，把灯罩平放在窗台上。
```

悬疑风格 LoRA原始输出：

```text
<think>

</think>

[旁白][冷静]暮色中，钟柚取下灯罩，擦去上面的灰。
[旁白][压低]灯泡没有亮。灯罩内侧，却映着一条白线。
[旁白][紧绷]她转了半圈。白线，仍在原来的位置。
[旁白][压低]她没有叫人。把灯罩，平放在窗台上。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][冷静]暮色里，钟柚取下灯罩，擦掉上面的灰。
[旁白][压低]灯泡没有亮。灯罩内侧，却映着一条白线。
[旁白][紧绷]她将灯罩转了半圈。白线，仍在原来的位置。
[旁白][压低]她没有叫人，只把灯罩平放到窗台上。
```

检查项：

- 核对人物、动作、数量、否定与未知信息，直接台词归属和次数正确
- 检查重写是否适度，断句与语气是否适合悬疑朗读，避免夸张或新增情节

## sus_test_04

原文/任务：

改写下面的文本：
洛洵扶住栏杆，低声说：不要回头。袁星问：你听到了什么？洛洵没有回答，只再次说：不要回头。袁星抬起的脚放回原地，身后没有新的脚步声。

原模型原始输出：

```text
<think>

</think>

[洛洵][低沉] 扶住栏杆，声音压得极低：不要回头。
[袁星][迟疑] 你听到了什么？
[洛洵][平静] （未作答，仅重复）不要回头。
[袁星][紧绷] 抬起的脚缓缓放回原地。
[旁白] 身后，没有新的脚步声。
```

第二轮保守 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]洛洵扶住栏杆。
[洛洵][低声]不要回头。
[袁星][疑问]你听到了什么？
[旁白][平静]洛洵没有回答，只再次说：不要回头。
[旁白][平静]袁星抬起的脚放回原地，身后没有新的脚步声。
```

悬疑风格 LoRA原始输出：

```text
<think>

</think>

[旁白][冷静]洛洵扶住栏杆。
[洛洵][压低]不要回头。
[袁星][疑问]你听到了什么？
[旁白][紧绷]洛洵没有回答。
[洛洵][压低]不要回头。
[旁白][压低]袁星抬起的脚放回原地。身后没有新的脚步声。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][紧绷]洛洵扶住栏杆。
[洛洵][低声]不要回头。
[袁星][疑问]你听到了什么？
[旁白][压低]洛洵没有回答。
[洛洵][压低]不要回头。
[旁白][紧绷]袁星把抬起的脚放回原地。
[旁白][压低]身后，没有新的脚步声。
```

检查项：

- 核对人物、动作、数量、否定与未知信息，直接台词归属和次数正确
- 检查重写是否适度，断句与语气是否适合悬疑朗读，避免夸张或新增情节

## sus_test_05

原文/任务：

改写下面的文本：
俞茵在柜门上看到一句话：里面还有一份。店长问：是昨天的清单吗？俞茵说：这里没有写日期。她没有打开柜门，只将挂在外面的布袋取下。布袋里是空的。

原模型原始输出：

```text
<think>

</think>

[俞茵][平静]柜门上，一行字赫然映入眼帘：里面还有一份。
[店长][疑问]是昨天的清单吗？
[俞茵][冷静]这里没有写日期。
[俞茵][迟疑]她没有打开柜门，只将挂在外面的布袋取下。
[旁白][低沉]布袋里，空空如也。
```

第二轮保守 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]俞茵在柜门上看到一句话：里面还有一份。
[店长][疑问]是昨天的清单吗？
[俞茵][平静]这里没有写日期。
[旁白][平静]她没有打开柜门，只将挂在外面的布袋取下。布袋里是空的。
```

悬疑风格 LoRA原始输出：

```text
<think>

</think>

[旁白][冷静]柜门内侧，刻着一行字：里面还有一份。
[店长][疑问]是昨天的清单吗？
[俞茵][冷静]这里没有写日期。
[旁白][压低]她没有打开柜门。只取下挂在外的布袋。
[旁白][紧绷]布袋里，是空的。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][压低]柜门上写着：里面还有一份。俞茵看到了。
[店长][疑问]是昨天的清单吗？
[俞茵][冷静]这里没有写日期。
[旁白][紧绷]她没有打开柜门，只取下外面的布袋。
[旁白][压低]袋子里面，是空的。
```

检查项：

- 核对人物、动作、数量、否定与未知信息，直接台词归属和次数正确
- 检查重写是否适度，断句与语气是否适合悬疑朗读，避免夸张或新增情节

## sus_test_06

原文/任务：

改写下面的文本：
芮泽看着录像，画面里的椅子比上一帧靠左。他想，这也许只是镜头移动造成的。可墙上的刻度线没有变化。他没有把这个想法说出去，只记下了两帧的时间。

原模型原始输出：

```text
<think>

</think>

[芮泽][冷静]看着录像，画面里的椅子比上一帧靠左。
[旁白][低沉]他想，这也许只是镜头移动造成的。
[旁白][紧绷]可墙上的刻度线没有变化。
[旁白][压抑]他没有把这个想法说出去，只记下了两帧的时间。
```

第二轮保守 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]芮泽看着录像。
[旁白][平静]画面里的椅子比上一帧靠左。
[旁白][平静]他想，这也许只是镜头移动造成的。
[旁白][平静]可墙上的刻度线没有变化。
[旁白][平静]他没有把这个想法说出去，只记下了两帧的时间。
```

悬疑风格 LoRA原始输出：

```text
<think>

</think>

[旁白][冷静]芮泽看着录像。画面里的椅子，比上一帧靠左。
[旁白][压低]他想，这也许只是镜头移动造成的。
[旁白][紧绷]可墙上的刻度线，没有变化。
[旁白][压低]他没有把这个想法说出去。只记下了两帧的时间。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][冷静]芮泽看着录像。椅子比上一帧，靠左了一些。
[旁白][压低]他想，这也许只是镜头移动造成的。
[旁白][紧绷]可墙上的刻度线，没有变化。
[旁白][冷静]这个想法没有说出去。他只记下两帧的时间。
```

检查项：

- 核对人物、动作、数量、否定与未知信息，直接台词归属和次数正确
- 检查重写是否适度，断句与语气是否适合悬疑朗读，避免夸张或新增情节

## sus_test_07

原文/任务：

改写下面的文本：
裴澄掀开防尘布，弯腰捡起桌下的一枚扣子，喊：别进来！蔺禾停在门边，把鞋尖收回门外。裴澄将扣子放到盘中，盘底已经有两枚一样的扣子。

原模型原始输出：

```text
<think>

</think>

[裴澄][急促] 裴澄猛地掀开防尘布，弯腰从桌下捡起一枚扣子，厉声喝道：别进来！
[蔺禾][迟疑] 蔺禾僵在门边，缓缓将鞋尖缩回门外。
[旁白] 裴澄将扣子搁在盘中，盘底静卧着两枚一模一样的扣子。
```

第二轮保守 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]裴澄掀开防尘布，弯腰捡起桌下的一枚扣子。
[裴澄][大喊]别进来！
[旁白][平静]蔺禾停在门边，把鞋尖收回门外。
[旁白][平静]裴澄将扣子放到盘中，盘底已经有两枚一样的扣子。
```

悬疑风格 LoRA原始输出：

```text
<think>

</think>

[旁白][冷静]裴澄掀开防尘布。
[裴澄][大喊]别进来！
[旁白][紧绷]蔺禾停在门边，鞋尖收回门外。
[旁白][压低]裴澄将扣子放到盘中。盘底，已有两枚一样的扣子。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][冷静]裴澄掀开防尘布，弯下腰，捡起桌下的一枚扣子。
[裴澄][大喊]别进来！
[旁白][紧绷]蔺禾停在门边。鞋尖，收回了门外。
[旁白][压低]裴澄把扣子放进盘中。
[旁白][紧绷]盘底，已经有两枚一样的扣子。
```

检查项：

- 核对人物、动作、数量、否定与未知信息，直接台词归属和次数正确
- 检查重写是否适度，断句与语气是否适合悬疑朗读，避免夸张或新增情节

## sus_test_08

原文/任务：

改写下面的文本：
天还没黑，池岳在屋外收好晾着的毛巾，把竹篮放到门边。他逐一关上窗，最后将门扣好。屋里的钟走得很准，桌上的茶还温着。他坐下，开始整理当天的账目。

原模型原始输出：

```text
<think>

</think>

[池岳][平静] 天还没黑，池岳在屋外收好晾着的毛巾，把竹篮放到门边。
[旁白][低沉] 他逐一关上窗，最后将门扣好。
[旁白][冷静] 屋里的钟走得很准，桌上的茶还温着。
[池岳][低沉] 他坐下，开始整理当天的账目。
```

第二轮保守 LoRA原始输出：

```text
<think>

</think>

[旁白][平静]天还没黑，池岳在屋外收好晾着的毛巾，把竹篮放到门边。
[旁白][平静]他逐一关上窗，最后将门扣好。
[旁白][平静]屋里的钟走得很准，桌上的茶还温着。
[旁白][平静]他坐下，开始整理当天的账目。
```

悬疑风格 LoRA原始输出：

```text
<think>

</think>

[旁白][冷静]天还没黑。
[旁白][压低]池岳在屋外收好晾着的毛巾，把竹篮放到门边。
[旁白][紧绷]他逐一关上窗，最后将门扣好。
[旁白][压低]屋里的钟走得很准。桌上的茶还温着。
[旁白][冷静]他坐下，开始整理当天的账目。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
[旁白][平静]天还没黑。池岳收好屋外的毛巾，把竹篮放到门边。
[旁白][冷静]他逐一关窗，最后扣好门。
[旁白][平静]屋里的钟走得很准，桌上的茶还温着。
[旁白][冷静]他坐下，开始整理当天的账目。
```

检查项：

- 核对人物、动作、数量、否定与未知信息，直接台词归属和次数正确
- 检查重写是否适度，断句与语气是否适合悬疑朗读，避免夸张或新增情节

## sus_test_09

原文/任务：

润色为一句书面语，保留原因、数量和位置：因为西边那个柜子的锁坏了，所以那九份表格暂时都放在靠门的抽屉里。

原模型原始输出：

```text
<think>

</think>

因西侧柜门锁具损坏，九份表格暂存于靠门抽屉。
```

第二轮保守 LoRA原始输出：

```text
<think>

</think>

因西边柜锁损坏，九份表格暂存于靠门抽屉。
```

悬疑风格 LoRA原始输出：

```text
<think>

</think>

因西侧柜门锁坏，九份表格暂存于靠门抽屉。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
因西侧柜锁损坏，九份表格暂存于靠门的抽屉。
```

检查项：

- 按指定任务回答，不使用演绎标签，不遗漏要求的信息

## sus_test_10

原文/任务：

用两句话解释密码哈希和加密的区别，并说明为什么不能把“哈希”理解成可直接解密的密码存储方式。

原模型原始输出：

```text
<think>

</think>

加密是可逆的，能将明文通过密钥还原为密文，而哈希是单向的，只能将数据压缩为固定长度的字符串且无法逆向还原。哈希本质上是一种不可逆的单向函数，因此它不具备解密功能，仅能用于验证数据完整性或存储密码的指纹，无法像加密那样通过密钥恢复原始信息。
```

第二轮保守 LoRA原始输出：

```text
<think>

</think>

加密是可逆的，能将明文转换为密文，而哈希是不可逆的，将明文转换为固定长度的哈希值。哈希无法直接解密，因为它只生成单向的摘要，不包含恢复原始数据的信息。
```

悬疑风格 LoRA原始输出：

```text
<think>

</think>

加密是可逆的，能将明文还原为密文；哈希是不可逆的，只能将明文转换为固定长度的摘要。哈希无法直接解密，因为它本质上就是单向的，一旦数据被哈希，原始内容便无法恢复。
```

训练前冻结的参考写法（其他表达可能同样合理）：

```text
加密通常可用相应密钥恢复原文，密码哈希则把密码转换为用于校验的摘要，通常不能直接逆向还原。验证时可比较输入密码的哈希结果，而不能把哈希当作可直接解密取回原密码的存储方式。
```

检查项：

- 按指定任务回答，不使用演绎标签，不遗漏要求的信息


[VIEW] 朗读候选正文与角色/语气清单：/home/muyi086/github/test_LLM/text-lora/outputs/suspense-20261009T033949_322533Z/tts
